# Incident Tickets — Cleaning

- Every missing value in **every field** is replaced with **"Unknown"**  
- Feature engineering(date parts + duration)  
- Data validation   
- Cleaned CSV exported

## 1. Imports

In [1]:
import re
import numpy as np
import pandas as pd

## 2. Load Dataset

In [4]:
ls

commands.sh          index_data.py        README.md
data/                instructions.txt     requirements.txt
data_cleaning.ipynb  LICENSE              visualizations/


In [5]:
csv_path = "data/dummy_incident_tickets.csv"
df = pd.read_csv(csv_path, low_memory=False)
print("Loaded shape:", df.shape)
df.head()

Loaded shape: (100, 30)


,Ticket Class,Ticket Priority,Ticket Number,Ticket Status,Ticket Status Original,Opened Date,Hostname,OS Type,Ticket Summary,Queue ID,...,Autogenerated,Automation Engine,Business Application,Closure Code,Sub Category,Target Finish Date,Time to resolve (min),Category,Alert Key,Comments & Work Notes
0,INCIDENT,3,INCEU1993810,CLOSED,CLOSED,2025-01-02T08:25:39,aks-sr-aks-prd,NaN,aks-sr-aks-prd issue detected and resolved aut...,K-PLTools-AIOPs-IN,...,N,Y,NaN,Email,NaN,2025-01-02T09:18:39,188,NaN,NaN,NaN
1,INCIDENT,3,INCEU1914165,CLOSED,CLOSED,2025-01-01T08:40:44,uatools-next-eu02,NaN,uatools-next-eu02 issue detected and resolved ...,K-PLTools-AIOPs-EU,...,Y,Y,NaN,Email,NaN,2025-01-01T09:16:44,56,NaN,NaN,NaN
2,INCIDENT,3,INCEU1930926,CLOSED,CLOSED,2025-01-06T03:05:43,uatools-next-eu02,NaN,uatools-next-eu02 issue detected and resolved ...,K-PLTools-AIOPs-IN,...,N,Y,NaN,Email,NaN,2025-01-06T04:49:43,23,NaN,NaN,NaN
3,INCIDENT,3,INCEU1955082,CLOSED,CLOSED,2025-01-04T05:02:50,org.logstash.logstash,NaN,org.logstash.logstash issue detected and resol...,K-PLTools-AIOPs-EU,...,Y,Y,NaN,Email,NaN,2025-01-04T05:59:50,141,NaN,NaN,NaN
4,INCIDENT,3,INCEU1919116,CLOSED,CLOSED,2025-01-01T13:20:50,aks-sr-aks-prd,NaN,aks-sr-aks-prd issue detected and resolved aut...,K-PLTools-AIOPs-EU,...,N,Y,NaN,Email,NaN,2025-01-01T16:23:50,97,NaN,NaN,NaN


## 3. Initial Data Information

In [6]:
print("Columns:", list(df.columns))
print("\nDtypes:")
print(df.dtypes)
print("\nNull counts:")
print(df.isna().sum())

Columns: ['Ticket Class', 'Ticket Priority', 'Ticket Number', 'Ticket Status', 'Ticket Status Original', 'Opened Date', 'Hostname', 'OS Type', 'Ticket Summary', 'Queue ID', 'Ticket Resolved Date', 'Resolution Code', 'Resolution Text', 'Ticket Closed Date', 'Call code', 'Reported By', 'Executed Automata', 'Recommended Automata', 'Actionable', 'Assignment Queue', 'Autogenerated', 'Automation Engine', 'Business Application', 'Closure Code', 'Sub Category', 'Target Finish Date', 'Time to resolve (min)', 'Category', 'Alert Key', 'Comments & Work Notes']

Dtypes:
Ticket Class               object
Ticket Priority             int64
Ticket Number              object
Ticket Status              object
Ticket Status Original     object
Opened Date                object
Hostname                   object
OS Type                   float64
Ticket Summary             object
Queue ID                   object
Ticket Resolved Date       object
Resolution Code            object
Resolution Text            o

## 4. Drop Empty columns

In [9]:
df = df.dropna(axis=1, how='all')
print("\nNull counts:")
print(df.isna().sum())


Null counts:
Ticket Class              0
Ticket Priority           0
Ticket Number             0
Ticket Status             0
Ticket Status Original    0
Opened Date               0
Hostname                  0
Ticket Summary            0
Queue ID                  0
Ticket Resolved Date      0
Resolution Code           0
Resolution Text           0
Ticket Closed Date        0
Call code                 0
Reported By               0
Executed Automata         0
Recommended Automata      0
Actionable                0
Assignment Queue          0
Autogenerated             0
Automation Engine         0
Closure Code              0
Target Finish Date        0
Time to resolve (min)     0
dtype: int64


## 5. Trim Whitespace in Text Columns

In [10]:
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype(str).str.strip()
df.head()

,Ticket Class,Ticket Priority,Ticket Number,Ticket Status,Ticket Status Original,Opened Date,Hostname,Ticket Summary,Queue ID,Ticket Resolved Date,...,Reported By,Executed Automata,Recommended Automata,Actionable,Assignment Queue,Autogenerated,Automation Engine,Closure Code,Target Finish Date,Time to resolve (min)
0,INCIDENT,3,INCEU1993810,CLOSED,CLOSED,2025-01-02T08:25:39,aks-sr-aks-prd,aks-sr-aks-prd issue detected and resolved aut...,K-PLTools-AIOPs-IN,2025-01-02T08:31:39,...,Automation Engine,Y,Y,Y,K-PLTools-AIOPs-IN,N,Y,Email,2025-01-02T09:18:39,188
1,INCIDENT,3,INCEU1914165,CLOSED,CLOSED,2025-01-01T08:40:44,uatools-next-eu02,uatools-next-eu02 issue detected and resolved ...,K-PLTools-AIOPs-EU,2025-01-01T09:03:44,...,Self-service,Y,Y,N,K-PLTools-AIOPs-EU,Y,Y,Email,2025-01-01T09:16:44,56
2,INCIDENT,3,INCEU1930926,CLOSED,CLOSED,2025-01-06T03:05:43,uatools-next-eu02,uatools-next-eu02 issue detected and resolved ...,K-PLTools-AIOPs-IN,2025-01-06T04:32:43,...,Self-service,Y,Y,Y,K-PLTools-AIOPs-IN,N,Y,Email,2025-01-06T04:49:43,23
3,INCIDENT,3,INCEU1955082,CLOSED,CLOSED,2025-01-04T05:02:50,org.logstash.logstash,org.logstash.logstash issue detected and resol...,K-PLTools-AIOPs-EU,2025-01-04T05:13:50,...,Self-service,Y,Y,N,K-PLTools-AIOPs-EU,Y,Y,Email,2025-01-04T05:59:50,141
4,INCIDENT,3,INCEU1919116,CLOSED,CLOSED,2025-01-01T13:20:50,aks-sr-aks-prd,aks-sr-aks-prd issue detected and resolved aut...,K-PLTools-AIOPs-EU,2025-01-01T16:09:50,...,Self-service,Y,Y,N,K-PLTools-AIOPs-EU,N,Y,Email,2025-01-01T16:23:50,97


## 6. Normalize Common Missing Tokens to NaN

In [11]:
common_missing = {"", "na", "n/a", "n.a.", "null", "none", "-", "--", "nan", "unknown"}
for col in df.columns:
    series = df[col]
    mask = series.notna()
    lowered = series[mask].astype(str).str.strip().str.lower()
    to_nan = lowered.isin(list(common_missing))
    df.loc[mask, col] = series[mask].where(~to_nan, np.nan)
df.head()

,Ticket Class,Ticket Priority,Ticket Number,Ticket Status,Ticket Status Original,Opened Date,Hostname,Ticket Summary,Queue ID,Ticket Resolved Date,...,Reported By,Executed Automata,Recommended Automata,Actionable,Assignment Queue,Autogenerated,Automation Engine,Closure Code,Target Finish Date,Time to resolve (min)
0,INCIDENT,3,INCEU1993810,CLOSED,CLOSED,2025-01-02T08:25:39,aks-sr-aks-prd,aks-sr-aks-prd issue detected and resolved aut...,K-PLTools-AIOPs-IN,2025-01-02T08:31:39,...,Automation Engine,Y,Y,Y,K-PLTools-AIOPs-IN,N,Y,Email,2025-01-02T09:18:39,188
1,INCIDENT,3,INCEU1914165,CLOSED,CLOSED,2025-01-01T08:40:44,uatools-next-eu02,uatools-next-eu02 issue detected and resolved ...,K-PLTools-AIOPs-EU,2025-01-01T09:03:44,...,Self-service,Y,Y,N,K-PLTools-AIOPs-EU,Y,Y,Email,2025-01-01T09:16:44,56
2,INCIDENT,3,INCEU1930926,CLOSED,CLOSED,2025-01-06T03:05:43,uatools-next-eu02,uatools-next-eu02 issue detected and resolved ...,K-PLTools-AIOPs-IN,2025-01-06T04:32:43,...,Self-service,Y,Y,Y,K-PLTools-AIOPs-IN,N,Y,Email,2025-01-06T04:49:43,23
3,INCIDENT,3,INCEU1955082,CLOSED,CLOSED,2025-01-04T05:02:50,org.logstash.logstash,org.logstash.logstash issue detected and resol...,K-PLTools-AIOPs-EU,2025-01-04T05:13:50,...,Self-service,Y,Y,N,K-PLTools-AIOPs-EU,Y,Y,Email,2025-01-04T05:59:50,141
4,INCIDENT,3,INCEU1919116,CLOSED,CLOSED,2025-01-01T13:20:50,aks-sr-aks-prd,aks-sr-aks-prd issue detected and resolved aut...,K-PLTools-AIOPs-EU,2025-01-01T16:09:50,...,Self-service,Y,Y,N,K-PLTools-AIOPs-EU,N,Y,Email,2025-01-01T16:23:50,97


## 7. Date-like Columns

In [12]:
date_like_cols = []
name_pattern = re.compile(r"(date|time|opened|closed|created|updated|resolved|sla)", re.I)
for c in df.columns:
    if name_pattern.search(c):
        date_like_cols.append(c)
    else:
        sample = df[c].dropna().astype(str).head(50)
        if not sample.empty:
            parsed = pd.to_datetime(sample, errors="coerce", infer_datetime_format=True)
            if parsed.notna().mean() >= 0.6:
                date_like_cols.append(c)

seen = set(); detected_dates = []
for c in date_like_cols:
    if c not in seen:
        detected_dates.append(c); seen.add(c)
detected_dates

/var/folders/p2/0by8fqfj559cf4qzzt1g5jym0000gn/T/ipykernel_37764/542499605.py:9: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(sample, errors="coerce", infer_datetime_format=True)
/var/folders/p2/0by8fqfj559cf4qzzt1g5jym0000gn/T/ipykernel_37764/542499605.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(sample, errors="coerce", infer_datetime_format=True)
/var/folders/p2/0by8fqfj559cf4qzzt1g5jym0000gn/T/ipykernel_37764/542499605.py:9: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see 

['Opened Date',
 'Ticket Resolved Date',
 'Ticket Closed Date',
 'Target Finish Date',
 'Time to resolve (min)']

## 8. Parse Datetime(`*_dt`)

In [13]:
for c in detected_dates:
    dt_col = f"{c}_dt"
    df[dt_col] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
df.filter(regex="_dt$").head()

/var/folders/p2/0by8fqfj559cf4qzzt1g5jym0000gn/T/ipykernel_37764/1846944531.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[dt_col] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/var/folders/p2/0by8fqfj559cf4qzzt1g5jym0000gn/T/ipykernel_37764/1846944531.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[dt_col] = pd.to_datetime(df[c], errors="coerce", infer_datetime_format=True)
/var/folders/p2/0by8fqfj559cf4qzzt1g5jym0000gn/T/ipykernel_37764/1846944531.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated 

,Opened Date_dt,Ticket Resolved Date_dt,Ticket Closed Date_dt,Target Finish Date_dt,Time to resolve (min)_dt
0,2025-01-02 08:25:39,2025-01-02 08:31:39,2025-01-02 09:18:39,2025-01-02 09:18:39,1970-01-01 00:00:00.000000188
1,2025-01-01 08:40:44,2025-01-01 09:03:44,2025-01-01 09:16:44,2025-01-01 09:16:44,1970-01-01 00:00:00.000000056
2,2025-01-06 03:05:43,2025-01-06 04:32:43,2025-01-06 04:49:43,2025-01-06 04:49:43,1970-01-01 00:00:00.000000023
3,2025-01-04 05:02:50,2025-01-04 05:13:50,2025-01-04 05:59:50,2025-01-04 05:59:50,1970-01-01 00:00:00.000000141
4,2025-01-01 13:20:50,2025-01-01 16:09:50,2025-01-01 16:23:50,2025-01-01 16:23:50,1970-01-01 00:00:00.000000097


## 9. Feature Engineering

In [14]:
# Extract Y/M/D from *_dt columns
dt_cols = [c for c in df.columns if c.endswith("_dt")]
for c in dt_cols:
    base = c[:-3]
    df[f"{base}_year"] = df[c].dt.year
    df[f"{base}_month"] = df[c].dt.month
    df[f"{base}_day"] = df[c].dt.day

# Incident duration if open/close detected
open_candidates = [c for c in detected_dates if re.search(r"(open|created|start)", c, re.I)]
close_candidates = [c for c in detected_dates if re.search(r"(close|resolved|end)", c, re.I)]
if open_candidates and close_candidates:
    open_dt = f"{open_candidates[0]}_dt"
    close_dt = f"{close_candidates[0]}_dt"
    if open_dt in df.columns and close_dt in df.columns:
        df["incident_duration_hours"] = (df[close_dt] - df[open_dt]).dt.total_seconds() / 3600.0

df.head()

,Ticket Class,Ticket Priority,Ticket Number,Ticket Status,Ticket Status Original,Opened Date,Hostname,Ticket Summary,Queue ID,Ticket Resolved Date,...,Ticket Closed Date_year,Ticket Closed Date_month,Ticket Closed Date_day,Target Finish Date_year,Target Finish Date_month,Target Finish Date_day,Time to resolve (min)_year,Time to resolve (min)_month,Time to resolve (min)_day,incident_duration_hours
0,INCIDENT,3,INCEU1993810,CLOSED,CLOSED,2025-01-02T08:25:39,aks-sr-aks-prd,aks-sr-aks-prd issue detected and resolved aut...,K-PLTools-AIOPs-IN,2025-01-02T08:31:39,...,2025,1,2,2025,1,2,1970,1,1,0.100000
1,INCIDENT,3,INCEU1914165,CLOSED,CLOSED,2025-01-01T08:40:44,uatools-next-eu02,uatools-next-eu02 issue detected and resolved ...,K-PLTools-AIOPs-EU,2025-01-01T09:03:44,...,2025,1,1,2025,1,1,1970,1,1,0.383333
2,INCIDENT,3,INCEU1930926,CLOSED,CLOSED,2025-01-06T03:05:43,uatools-next-eu02,uatools-next-eu02 issue detected and resolved ...,K-PLTools-AIOPs-IN,2025-01-06T04:32:43,...,2025,1,6,2025,1,6,1970,1,1,1.450000
3,INCIDENT,3,INCEU1955082,CLOSED,CLOSED,2025-01-04T05:02:50,org.logstash.logstash,org.logstash.logstash issue detected and resol...,K-PLTools-AIOPs-EU,2025-01-04T05:13:50,...,2025,1,4,2025,1,4,1970,1,1,0.183333
4,INCIDENT,3,INCEU1919116,CLOSED,CLOSED,2025-01-01T13:20:50,aks-sr-aks-prd,aks-sr-aks-prd issue detected and resolved aut...,K-PLTools-AIOPs-EU,2025-01-01T16:09:50,...,2025,1,1,2025,1,1,1970,1,1,2.816667


## 10. Replace **Every** Missing Value with `"Unknown"`

In [15]:
df = df.where(df.notna(), "Unknown")
df.head()

,Ticket Class,Ticket Priority,Ticket Number,Ticket Status,Ticket Status Original,Opened Date,Hostname,Ticket Summary,Queue ID,Ticket Resolved Date,...,Ticket Closed Date_year,Ticket Closed Date_month,Ticket Closed Date_day,Target Finish Date_year,Target Finish Date_month,Target Finish Date_day,Time to resolve (min)_year,Time to resolve (min)_month,Time to resolve (min)_day,incident_duration_hours
0,INCIDENT,3,INCEU1993810,CLOSED,CLOSED,2025-01-02T08:25:39,aks-sr-aks-prd,aks-sr-aks-prd issue detected and resolved aut...,K-PLTools-AIOPs-IN,2025-01-02T08:31:39,...,2025,1,2,2025,1,2,1970,1,1,0.100000
1,INCIDENT,3,INCEU1914165,CLOSED,CLOSED,2025-01-01T08:40:44,uatools-next-eu02,uatools-next-eu02 issue detected and resolved ...,K-PLTools-AIOPs-EU,2025-01-01T09:03:44,...,2025,1,1,2025,1,1,1970,1,1,0.383333
2,INCIDENT,3,INCEU1930926,CLOSED,CLOSED,2025-01-06T03:05:43,uatools-next-eu02,uatools-next-eu02 issue detected and resolved ...,K-PLTools-AIOPs-IN,2025-01-06T04:32:43,...,2025,1,6,2025,1,6,1970,1,1,1.450000
3,INCIDENT,3,INCEU1955082,CLOSED,CLOSED,2025-01-04T05:02:50,org.logstash.logstash,org.logstash.logstash issue detected and resol...,K-PLTools-AIOPs-EU,2025-01-04T05:13:50,...,2025,1,4,2025,1,4,1970,1,1,0.183333
4,INCIDENT,3,INCEU1919116,CLOSED,CLOSED,2025-01-01T13:20:50,aks-sr-aks-prd,aks-sr-aks-prd issue detected and resolved aut...,K-PLTools-AIOPs-EU,2025-01-01T16:09:50,...,2025,1,1,2025,1,1,1970,1,1,2.816667


## 11. Remove Exact Duplicate Rows

In [16]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows")

Removed 0 duplicate rows


## 12. Data Validation

In [17]:
print("Final shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nNull counts (should all be 0):")
print(df.isna().sum())
print("\nDuplicates:", df.duplicated().sum())

Final shape: (100, 45)

Dtypes:
Ticket Class                           object
Ticket Priority                         int64
Ticket Number                          object
Ticket Status                          object
Ticket Status Original                 object
Opened Date                            object
Hostname                               object
Ticket Summary                         object
Queue ID                               object
Ticket Resolved Date                   object
Resolution Code                        object
Resolution Text                        object
Ticket Closed Date                     object
Call code                              object
Reported By                            object
Executed Automata                      object
Recommended Automata                   object
Actionable                             object
Assignment Queue                       object
Autogenerated                          object
Automation Engine                      object
Cl

## 13. Export Cleaned Dataset

In [18]:
out_path = "data/dummy_incident_tickets_cleaned.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: data/dummy_incident_tickets_cleaned.csv
